# 製程能力分析：Cp, Cpk, Pp, Ppk

> **課程定位**：實務應用篇（5/6）  
> **前置課程**：[04_計數型管制圖_P_Chart](./04_計數型管制圖_P_Chart.ipynb)  
> **學習路徑**：01 概論 → 02 X-bar & R → 03 I-MR → 04 P Chart → **05 製程能力** → 06 綜合案例

## 學習目標
- 區分「製程穩定性」與「製程能力」兩個不同問題
- 理解規格界限（USL/LSL）與管制界限（UCL/LCL）的差異
- 掌握 Cp、Cpk、Pp、Ppk 的計算與解讀
- 能夠從製程能力指數推導預期不良率（PPM）

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Microsoft JhengHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

## 1. 管制（穩定性）vs. 能力（符合規格）

這是兩個**完全不同**的問題：

|  | 管制（Control） | 能力（Capability） |
|--|-----------------|-------------------|
| 問題 | 製程**穩定嗎**？ | 製程**夠好嗎**？ |
| 比較基準 | 管制界限（UCL/LCL） | 規格界限（USL/LSL） |
| 界限來源 | **數據計算**（±3σ） | **客戶/工程需求** |
| 工具 | 控制圖 | 能力指數 Cp/Cpk |

### 四種情境

| | 製程在管制中 | 製程不在管制中 |
|---|---|---|
| **有能力** | 理想狀態 | 暫時符合，但不可靠 |
| **無能力** | 穩定但不符規格 | 最差情況 |

> **知識補給站**  
> 進行能力分析**前**，必須先確認製程在統計管制中（使用控制圖）。對一個不穩定的製程計算 Cp/Cpk 是**沒有意義的**。

## 2. 規格界限 vs. 管制界限

這是初學者最常混淆的概念：

| | 規格界限 (USL/LSL) | 管制界限 (UCL/LCL) |
|--|--------------------|--------------------||
| 設定者 | 客戶 / 工程設計 | 製程數據計算 |
| 意義 | 產品**允許範圍** | 製程**自然波動範圍** |
| 可否調整 | 需與客戶協商 | 隨製程改善而改變 |
| 出現位置 | 能力分析 | 控制圖 |

$$\text{規格寬度} = USL - LSL$$
$$\text{製程寬度} = 6\sigma$$

**能力分析的本質**：比較規格寬度與製程寬度的關係。

## 3. Cp 與 Cpk 公式推導

### Cp（製程潛在能力指數）

$$C_p = \frac{USL - LSL}{6\sigma}$$

- 僅考慮**散布**，不考慮**偏心**
- Cp = 1 表示製程寬度恰好等於規格寬度
- Cp > 1 表示製程寬度小於規格寬度（有餘裕）

### Cpk（製程實際能力指數）

$$C_{pk} = \min\left(\frac{USL - \bar{X}}{3\sigma}, \;\frac{\bar{X} - LSL}{3\sigma}\right)$$

- 同時考慮**散布**和**偏心**
- Cpk ≤ Cp（永遠成立）
- Cpk = Cp 表示製程完全置中

### 解讀標準

| Cpk 值 | 等級 | 意義 | 預估 PPM |
|--------|------|------|----------|
| < 1.00 | 不合格 | 製程能力不足 | > 2,700 |
| 1.00 ~ 1.33 | 勉強合格 | 需要持續改善 | 64 ~ 2,700 |
| 1.33 ~ 1.67 | 合格 | 一般品質要求 | 0.6 ~ 64 |
| ≥ 1.67 | 優良 | 六標準差水準接近 | < 0.6 |
| ≥ 2.00 | 卓越 | 六標準差目標 | < 0.002 |

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

scenarios = [
    {'title': 'Cp=1.0, 置中\n（剛好符合規格）', 'mean': 100, 'sigma': 1.667, 'usl': 105, 'lsl': 95},
    {'title': 'Cp=2.0, 置中\n（六標準差水準）', 'mean': 100, 'sigma': 0.833, 'usl': 105, 'lsl': 95},
    {'title': 'Cp=1.33, 偏心\n（Cpk < Cp）', 'mean': 102, 'sigma': 1.25, 'usl': 105, 'lsl': 95},
    {'title': 'Cp=0.67, 置中\n（能力不足）', 'mean': 100, 'sigma': 2.5, 'usl': 105, 'lsl': 95},
]

for ax, s in zip(axes.flat, scenarios):
    x = np.linspace(s['mean'] - 4*s['sigma'], s['mean'] + 4*s['sigma'], 300)
    y = stats.norm.pdf(x, s['mean'], s['sigma'])
    
    ax.plot(x, y, 'b-', linewidth=2)
    ax.fill_between(x, y, alpha=0.3, color='steelblue')
    
    # 規格界限
    ax.axvline(x=s['usl'], color='red', linestyle='--', linewidth=2, label=f"USL={s['usl']}")
    ax.axvline(x=s['lsl'], color='red', linestyle='--', linewidth=2, label=f"LSL={s['lsl']}")
    
    # 計算 Cp, Cpk
    cp = (s['usl'] - s['lsl']) / (6 * s['sigma'])
    cpu = (s['usl'] - s['mean']) / (3 * s['sigma'])
    cpl = (s['mean'] - s['lsl']) / (3 * s['sigma'])
    cpk = min(cpu, cpl)
    
    # 著色超出規格的區域
    ax.fill_between(x, y, where=(x > s['usl']) | (x < s['lsl']), alpha=0.5, color='red')
    
    ax.set_title(f"{s['title']}\nCp={cp:.2f}, Cpk={cpk:.2f}", fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_ylim(bottom=0)

plt.suptitle('不同 Cp/Cpk 情境比較', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. 實務案例：晶圓蝕刻深度

### 情境描述

某晶圓廠的蝕刻站，蝕刻深度的規格為 **100 ± 5 μm**：
- USL = 105 μm
- LSL = 95 μm
- 目標值 = 100 μm

收集了 **200 個** 蝕刻深度的量測值進行能力分析。

In [ ]:
np.random.seed(42)

# 模擬數據：略微偏心
data = np.random.normal(loc=100.8, scale=1.3, size=200)
USL = 105
LSL = 95
target = 100

# Step 1: 描述性統計
mean = data.mean()
std = data.std(ddof=1)

print("=" * 50)
print("Step 1: 描述性統計")
print("=" * 50)
print(f"  樣本數:   {len(data)}")
print(f"  平均值:   {mean:.4f} \u03bcm")
print(f"  標準差:   {std:.4f} \u03bcm")
print(f"  最小值:   {data.min():.4f} \u03bcm")
print(f"  最大值:   {data.max():.4f} \u03bcm")
print(f"  目標值:   {target} \u03bcm")
print(f"  偏心量:   {mean - target:+.4f} \u03bcm")

# Step 2: 計算 Cp, Cpk
Cp = (USL - LSL) / (6 * std)
Cpu = (USL - mean) / (3 * std)
Cpl = (mean - LSL) / (3 * std)
Cpk = min(Cpu, Cpl)

print(f"\n{'=' * 50}")
print("Step 2: 製程能力指數")
print("=" * 50)
print(f"  Cp  = (USL-LSL)/(6\u03c3) = ({USL}-{LSL})/(6\u00d7{std:.4f}) = {Cp:.4f}")
print(f"  Cpu = (USL-X\u0304)/(3\u03c3) = ({USL}-{mean:.4f})/(3\u00d7{std:.4f}) = {Cpu:.4f}")
print(f"  Cpl = (X\u0304-LSL)/(3\u03c3) = ({mean:.4f}-{LSL})/(3\u00d7{std:.4f}) = {Cpl:.4f}")
print(f"  Cpk = min(Cpu, Cpl) = min({Cpu:.4f}, {Cpl:.4f}) = {Cpk:.4f}")

# Step 3: 預估 PPM
ppm_upper = stats.norm.sf((USL - mean) / std) * 1_000_000
ppm_lower = stats.norm.cdf((LSL - mean) / std) * 1_000_000
ppm_total = ppm_upper + ppm_lower

print(f"\n{'=' * 50}")
print("Step 3: 預估不良率 (PPM)")
print("=" * 50)
print(f"  超出 USL: {ppm_upper:.1f} PPM")
print(f"  低於 LSL: {ppm_lower:.1f} PPM")
print(f"  總計:     {ppm_total:.1f} PPM ({ppm_total/10000:.4f}%)")

In [ ]:
def process_capability_report(data, USL, LSL, target=None, title="製程能力分析報告"):
    """
    生成完整的製程能力分析報告
    
    Parameters:
    -----------
    data : array-like - 量測數據
    USL : float - 規格上限
    LSL : float - 規格下限
    target : float, optional - 目標值
    title : str - 報告標題
    """
    data = np.array(data, dtype=float)
    n = len(data)
    mean = data.mean()
    std = data.std(ddof=1)
    
    if target is None:
        target = (USL + LSL) / 2
    
    # 能力指數
    Cp = (USL - LSL) / (6 * std)
    Cpu = (USL - mean) / (3 * std)
    Cpl = (mean - LSL) / (3 * std)
    Cpk = min(Cpu, Cpl)
    
    # PPM
    ppm_upper = stats.norm.sf((USL - mean) / std) * 1_000_000
    ppm_lower = stats.norm.cdf((LSL - mean) / std) * 1_000_000
    ppm_total = ppm_upper + ppm_lower
    
    # 繪圖
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    # 左圖：直方圖 + 常態擬合 + 規格線
    ax = axes[0]
    ax.hist(data, bins=20, density=True, alpha=0.7, color='steelblue', edgecolor='white')
    
    x_fit = np.linspace(data.min() - std, data.max() + std, 300)
    y_fit = stats.norm.pdf(x_fit, mean, std)
    ax.plot(x_fit, y_fit, 'b-', linewidth=2, label='常態擬合')
    
    ax.axvline(x=USL, color='red', linestyle='--', linewidth=2, label=f'USL = {USL}')
    ax.axvline(x=LSL, color='red', linestyle='--', linewidth=2, label=f'LSL = {LSL}')
    ax.axvline(x=target, color='green', linestyle=':', linewidth=2, label=f'Target = {target}')
    ax.axvline(x=mean, color='orange', linestyle='-', linewidth=2, label=f'Mean = {mean:.2f}')
    
    # 著色超出規格
    ax.fill_between(x_fit, y_fit, where=(x_fit > USL), alpha=0.5, color='red')
    ax.fill_between(x_fit, y_fit, where=(x_fit < LSL), alpha=0.5, color='red')
    
    ax.set_title('數據分布與規格', fontsize=13)
    ax.set_xlabel('量測值')
    ax.set_ylabel('機率密度')
    ax.legend(fontsize=9)
    
    # 右圖：能力指數摘要
    ax2 = axes[1]
    ax2.axis('off')
    
    # 等級判定
    if Cpk >= 1.67:
        grade = "優良"
        color = 'green'
    elif Cpk >= 1.33:
        grade = "合格"
        color = 'blue'
    elif Cpk >= 1.00:
        grade = "勉強合格"
        color = 'orange'
    else:
        grade = "不合格"
        color = 'red'
    
    report_text = (
        f"\n"
        f"  ======================================\n"
        f"          製 程 能 力 分 析 摘 要       \n"
        f"  ======================================\n"
        f"                                        \n"
        f"  樣本數:    {n:>8d}                    \n"
        f"  平均值:    {mean:>8.4f}                \n"
        f"  標準差:    {std:>8.4f}                 \n"
        f"                                        \n"
        f"  USL:       {USL:>8.2f}                 \n"
        f"  LSL:       {LSL:>8.2f}                 \n"
        f"  Target:    {target:>8.2f}              \n"
        f"                                        \n"
        f"  --- 能力指數 ---                      \n"
        f"  Cp:        {Cp:>8.4f}                 \n"
        f"  Cpk:       {Cpk:>8.4f}                \n"
        f"  Cpu:       {Cpu:>8.4f}                \n"
        f"  Cpl:       {Cpl:>8.4f}                \n"
        f"                                        \n"
        f"  --- 預估不良率 ---                    \n"
        f"  PPM (>USL): {ppm_upper:>7.1f}         \n"
        f"  PPM (<LSL): {ppm_lower:>7.1f}         \n"
        f"  PPM Total:  {ppm_total:>7.1f}         \n"
        f"                                        \n"
        f"  等級:  {grade}                         \n"
        f"  ======================================\n"
    )
    
    ax2.text(0.1, 0.5, report_text, transform=ax2.transAxes,
             fontsize=11, family='monospace', verticalalignment='center',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    return {
        'Cp': Cp, 'Cpk': Cpk, 'Cpu': Cpu, 'Cpl': Cpl,
        'ppm_upper': ppm_upper, 'ppm_lower': ppm_lower, 'ppm_total': ppm_total,
        'mean': mean, 'std': std, 'grade': grade
    }

# 執行報告
report = process_capability_report(data, USL=105, LSL=95, target=100, 
                                    title="晶圓蝕刻深度 製程能力分析")

## 5. PPM 與 Sigma Level 對照

PPM（Parts Per Million）是每百萬個產品中的預期不良品數：

| Sigma Level | Cp | Cpk（置中） | PPM | 良率 |
|-------------|-----|------------|-----|------|
| 2σ | 0.67 | 0.67 | 45,500 | 95.45% |
| 3σ | 1.00 | 1.00 | 2,700 | 99.73% |
| 4σ | 1.33 | 1.33 | 63 | 99.9937% |
| 5σ | 1.67 | 1.67 | 0.57 | 99.99994% |
| 6σ | 2.00 | 2.00 | 0.002 | 99.9999998% |

> **知識補給站**  
> Motorola 的六標準差定義中，允許製程平均有 ±1.5σ 的偏移，因此 "Six Sigma" 實際對應 4.5σ 的 PPM（約 3.4 PPM），而非上表的理論值。

In [ ]:
print("Sigma Level vs. PPM 對照表")
print("=" * 60)
print(f"{'Sigma':>6} {'Cp':>8} {'PPM (理論)':>12} {'PPM (含1.5σ偏移)':>18} {'良率':>12}")
print("-" * 60)

for sigma_level in [2, 3, 4, 5, 6]:
    cp = sigma_level / 3
    ppm_theory = (1 - (stats.norm.cdf(sigma_level) - stats.norm.cdf(-sigma_level))) * 1e6
    ppm_shifted = (1 - (stats.norm.cdf(sigma_level - 1.5) - stats.norm.cdf(-sigma_level - 1.5))) * 1e6
    yield_rate = 1 - ppm_theory / 1e6
    
    print(f"{sigma_level:>5}\u03c3 {cp:>8.2f} {ppm_theory:>12.1f} {ppm_shifted:>18.1f} {yield_rate:>11.6%}")

## 6. Pp 與 Ppk（整體能力指標）

| | Cp / Cpk | Pp / Ppk |
|--|----------|----------|
| 標準差 | **子組內** σ（$\hat{\sigma} = \bar{R}/d_2$） | **整體** s（樣本標準差） |
| 反映 | 短期（within）能力 | 長期（overall）能力 |
| 使用時機 | 製程已在管制中 | 初步評估或長期追蹤 |

$$P_p = \frac{USL - LSL}{6s}, \quad P_{pk} = \min\left(\frac{USL - \bar{X}}{3s}, \frac{\bar{X} - LSL}{3s}\right)$$

**當 Cp ≈ Pp 時**：製程變異主要來自普通原因

**當 Cp >> Pp 時**：存在顯著的特殊原因變異，需要先用控制圖找出並消除

## 7. 本章小結

| 項目 | 內容 |
|------|------|
| 前提條件 | 製程必須先在統計管制中 |
| Cp | 潛在能力（不考慮偏心）= (USL-LSL)/(6σ) |
| Cpk | 實際能力（考慮偏心）= min(Cpu, Cpl) |
| 解讀標準 | Cpk ≥ 1.33 為一般品質要求 |
| PPM | 從 Cpk 推估每百萬不良品數 |
| 關鍵函數 | `process_capability_report()` |

---

## 課堂練習

### 基礎題
1. 某製程 X̄ = 50.0, σ = 1.0, USL = 53, LSL = 47。手算 Cp 和 Cpk。

### 進階題
2. 使用 `process_capability_report()` 對以下模擬數據進行分析：
   ```python
   np.random.seed(100)
   data = np.random.normal(loc=25.02, scale=0.08, size=150)
   USL, LSL = 25.30, 24.70
   ```
   解讀結果並給出改善建議。

### 挑戰題
3. 某製程目前 Cpk = 0.85（不合格）。計算：
   - 若保持 σ 不變，平均值需移動多少才能達到 Cpk = 1.33？
   - 若保持平均值不變，σ 需降低到多少才能達到 Cpk = 1.33？
   - 用 Python 求解兩種方案，並比較哪種更可行。

---

> **下一堂課**：[06_綜合應用_智慧製造案例](./06_綜合應用_智慧製造案例.ipynb) -- 整合所有 SPC 工具的完整案例